In [1]:
# Cell 1 — Imports and configuration (unchanged from original)
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

def find_repo_root(start=None):
    start = (start or Path.cwd()).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / 'config.py').exists() and (candidate / 'evaluation' / 'test_questions.json').exists():
            return candidate
    raise FileNotFoundError('Could not locate the repository root.')

REPO_ROOT = find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from vector_rag.pipeline     import VectorRAGPipeline
from vectorless_rag.pipeline import VectorlessRAGPipeline
from hybrid_rag.pipeline     import HybridRAGPipeline
from evaluation.evaluator    import run_evaluation, print_summary
from evaluation.ragas_evaluator import run_ragas_evaluation, merge_results, print_ragas_summary

RESULTS_DIR    = REPO_ROOT / 'evaluation' / 'results'
QUESTIONS_PATH = REPO_ROOT / 'evaluation' / 'test_questions.json'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

METHOD_ORDER  = ['vector', 'vectorless', 'hybrid']
METHOD_LABELS = {'vector': 'Vector RAG', 'vectorless': 'Vectorless RAG', 'hybrid': 'Hybrid RAG'}
METHOD_COLORS = {'vector': '#1f77b4', 'vectorless': '#ff7f0e', 'hybrid': '#2ca02c'}
RAGAS_METRIC_LABELS = {
    'answer_relevancy'    : 'Answer Relevancy',
    'faithfulness'        : 'Faithfulness',
    'context_precision'   : 'Contextual Precision',
    'context_recall'      : 'Contextual Recall',
    'contextual_relevancy': 'Contextual Relevancy\n(≈ context_precision)',
}

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'savefig.dpi': 160,
                     'axes.titlesize': 13, 'axes.labelsize': 11})

def save_figure(fig, filename):
    out = RESULTS_DIR / filename
    fig.savefig(out, dpi=160, bbox_inches='tight')
    return out

print('Imports ready.')

Imports ready.


In [2]:
# Cell 2 — Initialise pipelines
with open(QUESTIONS_PATH) as f:
    _q = json.load(f)
print(f'Loaded {len(_q["questions"])} questions for the benchmark.')

print('Initialising pipelines...')
vec    = VectorRAGPipeline()
vl     = VectorlessRAGPipeline()
hybrid = HybridRAGPipeline()
print('All three pipelines are ready.')

Loaded 20 questions for the benchmark.
Initialising pipelines...
🔧 Initialising Vector RAG Pipeline...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

✅ ChromaDB loaded — 8625 child vectors
Mistral client ready - model: mistral-medium-latest
✅ Vector RAG ready — 2467 parents in lookup

🔧 Initialising Vectorless RAG Pipeline...
✅ BM25 loaded — 8625 children
Mistral client already initialised - reusing
✅ Vectorless RAG ready — 8625 children, 2467 parents

🔧 Initialising Hybrid RAG Pipeline...
✅ ChromaDB loaded — 8625 child vectors
✅ BM25 loaded — 8625 children
Mistral client already initialised - reusing
✅ Hybrid RAG ready — 8625 vectors | 8625 BM25 children | 2467 parents

All three pipelines are ready.


In [3]:
# Cell 3 — Run judge evaluation
#
# capture_contexts=True tells run_evaluation() to also return a dict
# {(question_id, method): [context_str, ...]} so RAGAS can reuse the
# same retrieved contexts without re-running retrieval.
#
# The judge CSV is saved to: evaluation/results/three_way_judge_results.csv

judge_df, retrieved_contexts_map = run_evaluation(
    vector_pipeline     = vec,
    vectorless_pipeline = vl,
    hybrid_pipeline     = hybrid,
    results_filename    = 'three_way_judge_results.csv',
    capture_contexts    = True,          # NEW — required for RAGAS
)

print(f'Judge evaluation complete — {len(judge_df)} rows')
print_summary(judge_df)

Mistral judge ready - model: mistral-medium-latest

   PHASE 5 - EVALUATION (20 questions × 3 methods)
   Methods         : vector, vectorless, hybrid
   Generation limit: 18 RPM
   Judge limit     : 12 RPM
   Context capture : ON  (for RAGAS)



Questions:   0%|          | 0/20 [00:00<?, ?it/s]


[1/20] 1 - Amazon
🔁 Loading reranker: BAAI/bge-reranker-large


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

✅ Reranker ready
   hybrid pipeline error: Expected where value to be a str, int, float, or operator expression, got None in query.

===== DEBUG =====
MODEL: mistral-medium-latest
MESSAGE: ChatCompletionMessage(content='{"score": 5, "reason": "Exact figures and trajectory match context."}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
CONTENT TYPE: <class 'str'>
CONTENT: '{"score": 5, "reason": "Exact figures and trajectory match context."}'

   vector       score: 5/5 - Exact figures and trajectory match context.
   ⏳ Mistral judge rate limit — waiting 4.2s...

===== DEBUG =====
MODEL: mistral-medium-latest
MESSAGE: ChatCompletionMessage(content='{"score": 5, "reason": "Information not present in provided context"}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
CONTENT TYPE: <class 'str'>
CONTENT: '{"score": 5, "reason": "Information not present in provided context"}'

   vector

...

Questions:  95%|█████████▌| 19/20 [11:04<00:36, 36.22s/it]


===== DEBUG =====
MODEL: mistral-medium-latest
MESSAGE: ChatCompletionMessage(content='{"score": 1, "reason": "Answer is an error, not a response"}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
CONTENT TYPE: <class 'str'>
CONTENT: '{"score": 1, "reason": "Answer is an error, not a response"}'

   hybrid       score: 1/5 - Answer is an error, not a response

[20/20] 20 - NVIDIA
   hybrid pipeline error: Expected where value to be a str, int, float, or operator expression, got None in query.

===== DEBUG =====
MODEL: mistral-medium-latest
MESSAGE: ChatCompletionMessage(content='{"score": 5, "reason": "Exact figure and company supported by context."}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
CONTENT TYPE: <class 'str'>
CONTENT: '{"score": 5, "reason": "Exact figure and company supported by context."}'

   vector       score: 5/5 - Exact figure and company supported by context

Questions: 100%|██████████| 20/20 [11:37<00:00, 34.86s/it]


===== DEBUG =====
MODEL: mistral-medium-latest
MESSAGE: ChatCompletionMessage(content='{"score": 1, "reason": "Answer is an error, not a fact"}', refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None)
CONTENT TYPE: <class 'str'>
CONTENT: '{"score": 1, "reason": "Answer is an error, not a fact"}'

   hybrid       score: 1/5 - Answer is an error, not a fact

Results saved → C:\Users\nagal\Documents\AI\rag-benchmark\evaluation\results\three_way_judge_results.csv
Judge evaluation complete — 60 rows

   EVALUATION SUMMARY

-------------------------------------------------------
  Vector RAG
-------------------------------------------------------
  Avg judge score    : 4.40 / 5
  Pass rate  (>=3)   : 85.0%
  Company accuracy   : 70.0%
  Avg retrieval time : 0.9035s
  Avg rerank time    : 11.3859s
  Avg generation time: 1.69s
  Avg total latency  : 13.98s

-------------------------------------------------------
  Vectorless RAG
--------------------

method                 hybrid  vector  vectorless
category                                         
business_operations       1.0     5.0         5.0
company_performance       1.0     1.0         5.0
financial_metrics         1.0     4.5         4.5
risk_factors              1.0     5.0         4.0
strategic_development     1.0     5.0         4.0

-------------------------------------------------------
  Score by company
-------------------------------------------------------
method     hybrid  vector  vectorless
company                              
Amazon        1.0     4.2         5.0
Microsoft     1.0     5.0         4.2
NVIDIA        1.0     4.2         3.4
Netflix       1.0     4.2         5.0




In [4]:

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from evaluation.ragas_evaluator import run_ragas_evaluation
print("judge_df" in globals())
print("retrieved_contexts_map" in globals())

True
True


In [5]:
# Cell 4 — Run RAGAS evaluation
#
# RAGAS uses the Mistral judge API via an OpenAI-compatible wrapper.
# Make sure MISTRAL_JUDGE_API_KEY is set in your .env.
#
# Metrics computed:
#   answer_relevancy     → ResponseRelevancy: is the answer on-topic?
#   faithfulness         → Is every claim grounded in retrieved context?
#   context_precision    → Are the best chunks ranked first?
#   context_recall       → Does context cover the reference answer?
#                          (NaN when no reference_answer in test_questions.json)
#   contextual_relevancy → Same value as context_precision
#                          (no standalone metric in RAGAS 0.2.x)
#
# The RAGAS CSV is saved to: evaluation/results/three_way_ragas_results.csv
#
# RATE LIMIT TIP: batch_size=5 is safe with Mistral free-tier (1 RPM for judges).
# Increase to 10-15 if you have a higher-tier key.

import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.append(str(PROJECT_ROOT))
from evaluation.ragas_evaluator import run_ragas_evaluation
assert 'judge_df' in globals() and 'retrieved_contexts_map' in globals(), (
    "judge_df and retrieved_contexts_map must be defined first. "
    "Run the judge evaluation cell above before this RAGAS cell."
)
ragas_df = run_ragas_evaluation(
    judge_df               = judge_df,
    retrieved_contexts_map = retrieved_contexts_map,
    questions_path         = QUESTIONS_PATH,
    results_filename       = 'three_way_ragas_results.csv',
    batch_size             = 5,
)

print(f'RAGAS evaluation complete — {len(ragas_df)} rows')
print_ragas_summary(ragas_df)


   🚀 HIGH-SPEED ASYNC RAGAS EVALUATION
   Total Evaluation Rows : 60
   Judge LLM            : mistral-medium-latest
   Embeddings           : mistral-embed  (Mistral /v1/embeddings)
   Metrics (Base)       : answer_relevancy, faithfulness
   Metrics (With Refs)  : context_precision, context_recall

🔄 Running asynchronous evaluation for base metrics...


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]

🔄 Running asynchronous evaluation for reference metrics...


Evaluating:   0%|          | 0/120 [00:00<?, ?it/s]


🎉 RAGAS evaluation execution complete! Saved → C:\Users\nagal\Documents\AI\rag-benchmark\evaluation\results\three_way_ragas_results.csv
RAGAS evaluation complete — 60 rows

   RAGAS SUMMARY

  Vector RAG
  ------------------------------
  Answer Relevancy            : 0.7618
  Faithfulness                : 0.7368
  Context Precision           : 0.6286
  Context Recall              : 0.9286
  Contextual Relevancy        : 0.6286  (alias: context_precision)

  Vectorless RAG
  ------------------------------
  Answer Relevancy            : 0.5499
  Faithfulness                : 0.5702
  Context Precision           : 0.4464
  Context Recall              : 0.5769
  Contextual Relevancy        : 0.4464  (alias: context_precision)

  Hybrid RAG
  ------------------------------
  Answer Relevancy            : 0.6690
  Faithfulness                : 0.0250
  Context Precision           : 0.0000
  Context Recall              : 0.0000
  Contextual Relevancy        : 0.0000  (alias: context_precis

In [6]:
# import time

# # Create a unique filename for this run
# timestamp = time.strftime("%Y%m%d-%H%M%S")
# new_filename = f"three_way_ragas_results_{timestamp}.csv"
  
# ragas_df = run_ragas_evaluation(
#     judge_df               = judge_df,
#     retrieved_contexts_map = retrieved_contexts_map,
#     questions_path         = QUESTIONS_PATH,
#     results_filename       = new_filename  # Use the unique name
# )